# 1.从全连接层到卷积
我们之前讨论的多层感知机十分合适处理表格数据，其中行对应样本，列对应特征。对于表格数据，我们寻找的模式可能设计特征之间的交互，但是我们不能预先假设任何与特征交互相关的先验结构。此时，多层感知机可能是最好的选择，然而对于高维感知数据，这种缺少结构的网络可能会变得不实用。

例如在猫狗分类的例子中，假如有一个足够多的数据集并且带有标签，每张照片有百万像素，这意味着网络的每次输入都有$10^6$个维度。即使我们将隐藏层维度降低到1000，这个全连接层也将有$10^6 \times 10^3=10^9$个参数。想要训练这个模型基本不可能。

如果将分辨率降低到十万级像素，使用1000个隐藏单元的隐藏层可能不足以学习到良好的图像特征。然而，如今人类和机器都能很好地区分猫狗：**这是因为图像中本就具有丰富的结构，而这些结构可以被人类和机器学习模型使用。卷积神经网络是机器学习利用自然图像中一些已知结构的创造性方法。**

## 1.1 不变性
假如我们想从一张图中找到某个物体。合理的假设就是：无论用哪种方法找到这个物体，都应该和物体的位置无关。例如猪不会在天上飞，但是它如果出现在天空中，我们也应该识别出它是一个猪。**卷积神经网络正是将空间不变性的这一概念系统化，从而基于这个模型使用较少的参数来学习有用的表示。**

$\cdot$ **平移不变性** ：不管检测对象出现在哪个位置，神经网络的前面几层应该对相同的图像区域具有相似的反应，即“平移不变性”。</br>
$\cdot$ **局部性** ：神经网络的前面几层应该只探索输入图像中的局部区域，而不过度在意图像中相隔较远区域的关系，这就是“局部性”原则。最终，可以聚合这些局部特征，以在整张图象进行预测。

## 1.2 多层感知机的限制
多层感知机的输入是二维图像X，其隐藏表示H在数学上是一个矩阵，在代码中表示位二维张量。X和H具有相同的形状。为了方便理解，我们可以认为，无论是输入还是隐藏表示都具有空间结构。
</br>
使用$[X]_{i,j}$和$[H]_{i,j}$分别表示输入图像和隐藏表示中位置$(i,j)$处的像素。为了使每个隐藏神经元都能接收每个输入像素的信息，我们将参数从权重矩阵替换为四阶权重张量W。我们假设U包含偏置参数，我们可以将全连接层形式化地表示为：
$$
[H]_{i,j} = [U]_{i,j} + \sum_{k} \sum_{l}[W]_{i, j, k, l}[X]_{k, l} = [U]_{i, j} + \sum_{a} \sum_{b}[V]_{i,j,a,b}[X]_{i+a, j+b}
$$

$[W]_{i,j,k,l}$ 输出位置$(i, j)$，输入位置$(k, l)$，

第二个形式：这个变换只是把 k = i+a, l = j+b 代入，从而把权重索引改写为相对位移 (a, b)。

其中，从W到V的转换只是形式上的转换，因为在这两个四阶张量的元素之间存在一一对应的关系。我们只需重新索引下标$(k,l)$，使$k=i+a、l=j+b$，由此可得$[V]_{i, j, a, b} = [W]_{i, j, i+a, j+b}$。索引a和b通过在正偏移和负偏移之间移动覆盖了整张图像。 对于隐藏表示中任意给定位置$(i,j)$处的像素值$[H]_{i,j}$，可以通过在x中以$(i, j)$为中心对像素进行加权求和得到（就是因为每个像素都要和隐藏层的每个单元连接。），加权使用的权重为$[V]_{i,j,a,b}$。

### 1. 平移不变性
现在引用上述的第一个原则：平移不变性。这意味着检测对象在输入X中的平移，应该仅导致隐藏表示中的平移。也就是说，V和U实际上不依赖$(i,j)$的值，即$[V]_{i,j,a,b}=[V]_{a,b}$。并且U是一个常数，比如u。因此，我们可以简化为H定义为：
$$
[H]_{i,j} = u + \sum_{a} \sum_{b}[V]_{a,b}[X]_{i+a, i+b}
$$

平移不变性的意思是：</br>
图像里的一只猫，不管它在左上角还是右下角，识别它的方式应该一样。</br>
放到公式里：</br>
假设在位置 (5,5) 有一个边缘特征</br>
在位置 (10,10) 也有完全一样的边缘特征</br>
平移不变性要求：</br>
检测这个边缘所用的权重，不应该依赖于它出现在哪里。</br>
也就是说：</br>
在 (5,5) 处检测边缘：用权重 V[5,5, a,b]</br>
在 (10,10) 处检测边缘：用权重 V[10,10, a,b]

这就是卷积，我们使用系数$[V]_{a,b}$对位置$(i,j)$附近的像素$(i+a,i+b)$进行加权得到$[H]_{i,j}$。注意，$[V]_{a,b}$的系数比$[V]_{i,j,a,b}$少许多，因为前者不再依赖图像中的位置。

权重的量级从$H*W*H*W$变成了$H*W$：

数学上：效果当然不一致（从自由变量变成共享变量）。

物理/视觉上：因为图像内容具有平移不变性（物体出现在哪里都是那个物体），所以共享权重效果不仅一致，而且更好。

核心转变：从“每个位置学一套参数”变成了“学一个通用的特征检测器”。

一句话总结：共享权重（卷积）是通过牺牲“位置特异性”来换取“特征通用性”，这恰恰是计算机视觉成功的基石。